# Chapter 6 · The client and the transports

The server of Northside Veterinary Clinic with three transports: in memory, through stdio and over HTTP. We measure what each one costs, what happens when two receptionists book at the same time, and what happens when two servers publish a tool with the same name.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. The first two parts don't need it; the third one does.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. The data and the clinic server

The server is a separate program; it is saved in a file so that it can be launched as a process.

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # the versions the book was checked with

In [ ]:
"""Chapter 6: the client and the transports. The clinic server in memory, through stdio
and over HTTP; two receptionists at the same time; and two tools with the same name."""
import asyncio
import json
import re
import time
import subprocess
import sys
import urllib.error
import urllib.request
from contextlib import AsyncExitStack
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client, StdioServerParameters
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
MODEL = None                               # None: choose according to the hardware
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/en/"
TODAY = date(2026, 10, 20)                 # the date of the clinic's data


def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; if not, on the CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model

In [ ]:
import os
os.makedirs("data", exist_ok=True)
for name in ("pets", "owners", "history", "vaccines", "appointments", "pharmacy",
             "invoices", "vets", "prices"):
    if not os.path.exists(f"data/{name}.json"):
        urllib.request.urlretrieve(REPOSITORY + "data/" + f"{name}.json",
                                   f"data/{name}.json")

In [ ]:
%%writefile clinic_server.py
"""MCP server of Northside Veterinary Clinic: a reduced version of chapter 5's, as a
separate program, with the lookup and schedule tools and one more,
upcoming_appointments.

It can be used in three ways: import it and connect an in-memory client, launch it as
a process that talks through stdio, or launch it as an HTTP service that several
clients connect to at the same time.
Usage: python clinic_server.py [data_folder] [--http PORT]
"""
import argparse
import copy
import json
import unicodedata
from datetime import date
from pathlib import Path
from typing import Annotated

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field

TODAY = date(2026, 10, 20)                 # the date of the clinic's data


# --- The clinic's data -------------------------------------------------------

def load_data(folder="data"):
    """Loads the clinic's data from a folder and resets the schedule."""
    global PETS, OWNERS, HISTORY, VACCINES, APPOINTMENTS0, PHARMACY, INVOICES, VETS
    global TIMES
    read = lambda name: json.loads((Path(folder) / name).read_text(encoding="utf-8"))
    PETS, OWNERS = read("pets.json"), read("owners.json")
    HISTORY, VACCINES = read("history.json"), read("vaccines.json")
    APPOINTMENTS0, PHARMACY = read("appointments.json"), read("pharmacy.json")
    INVOICES, VETS = read("invoices.json"), read("vets.json")
    TIMES = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reset()


def reset():
    """Puts the clinic back as it was: the original schedule and no notices."""
    global APPOINTMENTS, NOTICES
    APPOINTMENTS, NOTICES = copy.deepcopy(APPOINTMENTS0), []


def strip_accents(text):
    """Lowercase and without accents, to compare names however they are written."""
    return "".join(c for c in unicodedata.normalize("NFD", str(text).lower())
                   if unicodedata.category(c) != "Mn").strip()


def age(birth_date):
    """Full years on the date of the data."""
    b = date.fromisoformat(birth_date)
    return TODAY.year - b.year - ((TODAY.month, TODAY.day) < (b.month, b.day))


def pet_summary(pid):
    """The essentials of a pet, without the full history."""
    p = PETS[pid]
    return {"name": p["name"], "species": p["species"], "breed": p["breed"],
            "age_years": age(p["birth_date"]), "current_weight_kg": p["weight_kg"],
            "allergies": p["allergies"] or "none known",
            "owner": OWNERS[p["owner"]]["name"]}


def vet_of(name):
    """The vet's id from their name or their id."""
    for v, n in VETS.items():
        if strip_accents(name) in (strip_accents(v), strip_accents(n)) or \
                strip_accents(name) in strip_accents(n):
            return v
    return None


def pet_of(name, owner=""):
    """The pet's id; a ToolError if there is none or there are several."""
    ids = [p for p, d in PETS.items() if strip_accents(name) == strip_accents(d["name"])
           and strip_accents(owner) in strip_accents(OWNERS[d["owner"]]["name"])]
    if not ids:
        raise ToolError(f"There is no pet called \"{name}\""
                        + (f" belonging to {owner}" if owner else "") + " at the clinic.")
    if len(ids) > 1:
        raise ToolError(f"There are {len(ids)} pets called \"{name}\"; give the "
                        "owner's name.")
    return ids[0]


def free(day, vet=""):
    """The free times of a day; a ToolError for a bad date, a weekend or a vet."""
    try:
        when = date.fromisoformat(day)
    except ValueError:
        raise ToolError(f"\"{day}\" is not a date; use YYYY-MM-DD.")
    if when.weekday() >= 5:
        raise ToolError(f"{day} is on a weekend and the clinic is closed.")
    v = vet_of(vet) if vet else None
    if vet and v is None:
        raise ToolError(f"There is no vet \"{vet}\". They are: "
                        + ", ".join(VETS.values()) + ".")
    taken = {(a["vet"], a["time"]) for a in APPOINTMENTS
             if a["date"] == day and a["status"] != "canceled"}
    return {"date": day, "free_times": {VETS[x]: [t for t in TIMES if (x, t) not in taken]
                                        for x in ([v] if v else VETS)}}


# --- The server --------------------------------------------------------------

Name = Annotated[str, Field(description="Name of the pet.")]
Owner = Annotated[str, Field(description="Owner's name; only if several pets have "
                                         "that name.")]
Date = Annotated[str, Field(description="YYYY-MM-DD.")]


def create_server():
    """The clinic's MCP server: the lookup and schedule tools."""
    server = MCPServer("clinic", instructions="Pets, vaccines, medical history and "
                       "schedule of Northside Veterinary Clinic.")

    @server.tool()
    def find_pet(name: Name, owner: Owner = "") -> dict:
        """A pet's data, by name: species, age, weight, allergies and owner."""
        return pet_summary(pet_of(name, owner))

    @server.tool()
    def vaccines(pet: Name, owner: Owner = "") -> dict:
        """A pet's vaccines: the next date of each one and whether it is overdue."""
        pid = pet_of(pet, owner)
        return {"pet": PETS[pid]["name"], "today": TODAY.isoformat(), "vaccines": [
            {"vaccine": v["vaccine"], "next": v["next"],
             "status": "overdue" if v["next"] < TODAY.isoformat() else "up to date"}
            for v in VACCINES[pid]]}

    @server.tool()
    def history(pet: Name, last: Annotated[int, Field(
            description="How many visits, from the most recent to the oldest.")] = 3,
                owner: Owner = "") -> dict:
        """A pet's last visits, starting with the most recent."""
        pid = pet_of(pet, owner)
        return {"pet": PETS[pid]["name"], "visits": [
            {k: v[k] for k in ("date", "reason", "treatment", "weight_kg")}
            for v in reversed(HISTORY[pid][-last:])]}

    @server.tool()
    def free_slots(date: Date, vet: Annotated[str, Field(
            description="Name of the vet; empty for all.")] = "") -> dict:
        """Free times in the schedule on a weekday, for one vet or for all."""
        return free(date, vet)

    @server.tool()
    def appointments_of(pet: Name, owner: Owner = "") -> dict:
        """A pet's appointments (past and scheduled), with their id."""
        pid = pet_of(pet, owner)
        return {"pet": PETS[pid]["name"], "today": TODAY.isoformat(), "appointments": [
            {"id": a["id"], "date": a["date"], "time": a["time"], "reason": a["reason"],
             "vet": VETS[a["vet"]], "status": a["status"]}
            for a in APPOINTMENTS if a["pet"] == pid]}

    @server.tool()
    def book_appointment(pet: Name,
                         vet: Annotated[str, Field(description="Name of the vet.")],
                         date: Date, time: Annotated[str, Field(
                             description="HH:MM, on the hour or half past.")],
                         reason: Annotated[str, Field(
                             description="Reason for the appointment.")],
                         owner: Owner = "") -> dict:
        """Books an appointment at a free time."""
        pid, v = pet_of(pet, owner), vet_of(vet)
        if v is None:
            raise ToolError(f"There is no vet \"{vet}\".")
        free(date, vet)
        if time not in TIMES:
            raise ToolError(f"\"{time}\" is not an appointment time.")
        if any(a["date"] == date and a["time"] == time and a["vet"] == v
               and a["status"] != "canceled" for a in APPOINTMENTS):
            raise ToolError(f"{VETS[v]} already has an appointment on {date} "
                            f"at {time}.")
        appointment = {"id": f"AP-{len(APPOINTMENTS0) + len(APPOINTMENTS) + 1:04d}",
                       "date": date, "time": time, "vet": v, "pet": pid,
                       "reason": reason, "status": "scheduled"}
        APPOINTMENTS.append(appointment)
        return {"booked": appointment["id"], "pet": PETS[pid]["name"], "vet": VETS[v],
                "date": date, "time": time}

    @server.tool()
    def cancel_appointment(appointment: Annotated[str, Field(
            description="Id of the appointment, like AP-0026 (given by "
                        "appointments_of).")]) -> dict:
        """Cancels a scheduled appointment."""
        for a in APPOINTMENTS:
            if a["id"] == appointment:
                if a["status"] != "scheduled":
                    raise ToolError(f"Appointment {appointment} is \"{a['status']}\".")
                a["status"] = "canceled"
                return {"canceled": appointment, "date": a["date"], "time": a["time"]}
        raise ToolError(f"There is no appointment {appointment}.")

    @server.tool()
    def upcoming_appointments() -> dict:
        """All the scheduled appointments in the schedule, to check what was booked."""
        return {"appointments": [{k: a[k] for k in ("id", "date", "time", "vet", "pet")}
                                 for a in APPOINTMENTS if a["status"] == "scheduled"]}

    return server


if __name__ == "__main__":
    arguments = argparse.ArgumentParser()
    arguments.add_argument("data", nargs="?", default="data")
    arguments.add_argument("--http", type=int, help="port; without it, the server "
                           "talks through stdio")
    a = arguments.parse_args()
    load_data(a.data)
    server = create_server()
    if a.http:
        server.run(transport="streamable-http", host="127.0.0.1", port=a.http)
    else:
        server.run()

In [ ]:
SERVER = Path("clinic_server.py")      # in the notebook, in the working folder
sys.path.insert(0, str(SERVER.parent))
import clinic_server as clinic  # noqa: E402

PORT = 8765

## 2. What each transport costs

In [ ]:
def in_memory():
    """A server in the same program: all the clients share its data."""
    return clinic.create_server()


def via_stdio():
    """Each client launches its own server process, with its own copy of the data."""
    return StdioServerParameters(command=sys.executable, args=[str(SERVER), str(DATA)])


def via_http():
    """The address of an HTTP server that is already running; every client talks to it."""
    return f"http://127.0.0.1:{PORT}/mcp"


def launch_http():
    """Launches the HTTP server as a process and waits until it answers."""
    process = subprocess.Popen([sys.executable, str(SERVER), str(DATA),
                                "--http", str(PORT)],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(100):
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/mcp", timeout=1)
        except urllib.error.HTTPError:          # it answers, even with an error: ready
            return process
        except OSError:
            time.sleep(0.2)
    raise RuntimeError("The HTTP server doesn't start.")


TRANSPORTS = {"in memory": in_memory, "stdio": via_stdio, "HTTP": via_http}


async def measure_transport(name, calls=100):
    """How long it takes to connect (introduction and list of tools) and each call."""
    start = time.perf_counter()
    async with Client(TRANSPORTS[name]()) as c:
        await c.list_tools()
        connect = time.perf_counter() - start
        start = time.perf_counter()
        for _ in range(calls):
            await c.call_tool("find_pet", {"name": "Lola"})
        call = (time.perf_counter() - start) / calls
    return {"connect_ms": round(connect * 1000, 1), "call_ms": round(call * 1000, 2)}

## 3. Two receptionists at the same time

In [ ]:
async def two_receptionists(name, rounds=6):
    """Two clients with the same transport. In each round, both look up Dr. Meyer's
    first free time on Monday the 26th and both book it; if the booking fails, the
    second one looks again and books the next one. At the end, we count the
    appointments that ended up at the same time with the same vet."""
    clinic.reset()
    target = TRANSPORTS[name]()
    bookings, rejections = [], 0
    async with Client(target) as amy, Client(target) as luke:
        for n in range(rounds):
            slots = []
            for c in (amy, luke):
                r = json.loads((await c.call_tool("free_slots", {
                    "date": "2026-10-26", "vet": "Meyer"})).content[0].text)
                slots.append(r["free_times"]["Dr. Paul Meyer"][0])
            for c, slot, pet in ((amy, slots[0], "Lola"), (luke, slots[1], "Toby")):
                request = {"pet": pet, "vet": "Meyer", "date": "2026-10-26",
                           "time": slot, "reason": f"checkup {n}"}
                r = await c.call_tool("book_appointment", request)
                if r.is_error:
                    rejections += 1
                    again = json.loads((await c.call_tool("free_slots", {
                        "date": "2026-10-26", "vet": "Meyer"})).content[0].text)
                    request["time"] = again["free_times"]["Dr. Paul Meyer"][0]
                    r = await c.call_tool("book_appointment", request)
                if not r.is_error:
                    bookings.append(request["time"])
    doubles = len(bookings) - len(set(bookings))
    return {"bookings": len(bookings), "rejected_and_retried": rejections,
            "times_booked_twice": doubles}


async def transports_experiment():
    """The cost of each transport and the two receptionists, without the model."""
    clinic.load_data(DATA)
    process = launch_http()
    try:
        results = {}
        for name in TRANSPORTS:
            results[name] = {**await measure_transport(name),
                             **await two_receptionists(name)}
            print(f"{name:12s}", results[name], flush=True)
    finally:
        process.terminate()
    return results

In [ ]:
transports = await transports_experiment()

## 4. Two tools with the same name

With the model. Two small servers use the same word, `search`, for different things.

In [ ]:
tok, model = load(MODEL)

In [ ]:
SYSTEM = (
    "You are the assistant of the staff of Northside Veterinary Clinic, a veterinary "
    "clinic in Portland, Oregon. Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in English, in one or two sentences.")


def generate(messages, schemas, max_tokens=400):
    """One answer from the model, and the input and output tokens."""
    text = tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    return (tok.decode(new, skip_special_tokens=True).strip(),
            inputs["input_ids"].shape[1], len(new))


def read_calls(answer):
    """The calls the model asks for, written between <tool_call> and </tool_call>."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"], "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}})
    return calls


def says(answer, *forms):
    """Does any of those forms appear in the answer?"""
    return any(re.search(f, answer or "", re.I) for f in forms)


def without_titles(schema):
    """The schema without the "title" keys that Pydantic adds (the model needs none)."""
    if isinstance(schema, dict):
        return {k: without_titles(v) for k, v in schema.items() if k != "title"}
    return schema


def compact(text):
    """A result's JSON on a single line, without the indentation the server sends."""
    try:
        return json.dumps(json.loads(text), ensure_ascii=False)
    except ValueError:
        return text


async def mcp_agent(task, host, max_steps=8, show=False):
    """Chapter 1's loop, with the tools of the MCP servers."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_in, n_out = generate(messages, host.schemas)
        calls = read_calls(answer)[:1]
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_in, "thought": thought,
                      "calls": []})
        if not calls:
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            text = await host.call(call["name"], call["arguments"])
            trace[-1]["calls"].append({**call, "result": text,
                                       "result_tokens": len(tok(text)["input_ids"])})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                args = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({args})")
                print(f"     → {text[:150]}")
    return None, trace

In [ ]:
record = MCPServer("clinic")


@record.tool()
def search(name: Annotated[str, Field(description="Name of the pet.")]) -> dict:
    """Looks up a pet of the clinic by name: species, breed, age, weight and allergies."""
    return clinic.pet_summary(clinic.pet_of(name))


pharmacy = MCPServer("pharmacy")


@pharmacy.tool(name="search")
def find_medicine(name: Annotated[str, Field(
        description="Name of the medicine.")]) -> dict:
    """Looks up a medicine of the pharmacy: presentation, species, dose and stock."""
    m = clinic.PHARMACY.get(clinic.strip_accents(name))
    if m is None:
        raise ToolError(f"We don't have \"{name}\".")
    return {"medicine": clinic.strip_accents(name), **{k: m[k] for k in (
        "presentation", "species", "dose_mg_kg", "stock")}}


class Host:
    """Chapter 4's host (compact), with two ways of gathering the tools: without
    namespaces, the last tool with a name replaces the previous ones; with namespaces,
    each tool is called server_tool."""

    def __init__(self, servers, namespaces=False):
        self.servers, self.namespaces = servers, namespaces

    async def __aenter__(self):
        self.stack = AsyncExitStack()
        self.route, self.schemas = {}, []
        for server in self.servers:
            client = await self.stack.enter_async_context(Client(server))
            for t in (await client.list_tools()).tools:
                name = f"{client.server_info.name}_{t.name}" if self.namespaces \
                    else t.name
                self.route[name] = (client, t.name)
                self.schemas.append({"type": "function", "function": {
                    "name": name, "description": t.description,
                    "parameters": without_titles(t.input_schema)}})
        return self

    async def __aexit__(self, *error):
        await self.stack.aclose()

    async def call(self, name, arguments):
        """Sends the call to the server of that name, with the tool's original name."""
        if name not in self.route:
            return json.dumps({"error": f"There is no tool {name}."},
                              ensure_ascii=False)
        client, original = self.route[name]
        r = await client.call_tool(original, arguments)
        return compact("\n".join(b.text for b in r.content if b.type == "text"))


TASKS = [
    ("kira_breed", "What breed is Kira?", lambda r: says(r, r"mixed")),
    ("toby_allergy", "Is Toby allergic to anything?", lambda r: says(r, "penicillin")),
    ("zeus_age", "How old is Zeus?", lambda r: says(r, r"\b4\b", r"\bfour\b")),
    ("max_weight", "How much does Max weigh?", lambda r: says(r, r"20\.7")),
    ("presentation", "In what presentation do we have meloxicam?",
     lambda r: says(r, "suspension")),
    ("species", "Which species is maropitant indicated for?",
     lambda r: says(r, r"\bdogs?\b") and not says(r, r"\bcats?\b")),
    ("stock", "How many units of enrofloxacin do we have left?",
     lambda r: says(r, rf"\b{clinic.PHARMACY['enrofloxacin']['stock']}\b")),
    ("dose_per_kg", "What is the dose per kilogram of gabapentin?",
     lambda r: says(r, rf"\b{clinic.PHARMACY['gabapentin']['dose_mg_kg']:g}\b"
                    .replace(".", r"\."))),
]


async def names_experiment():
    """The eight questions with two tools called search, without and with namespaces."""
    clinic.load_data(DATA)
    results = {}
    for name, namespaces in (("Without namespaces", False), ("With namespaces", True)):
        rows = []
        async with Host([record, pharmacy], namespaces) as host:
            for key, task, check in TASKS:
                answer, trace = await mcp_agent(task, host)
                rows.append({"task": key, "ok": bool(check(answer)), "answer": answer,
                             "calls": [(c["name"], c["arguments"]) for s in trace
                                       for c in s["calls"]]})
            names = [s["function"]["name"] for s in host.schemas]
        results[name] = {"ok": sum(r["ok"] for r in rows), "tools": names, "rows": rows}
        print(f"{name:26s}", results[name]["ok"], "of", len(TASKS), names, flush=True)
        print("    " + " ".join(("+" if r["ok"] else "-") + r["task"] for r in rows),
              flush=True)
    return results

In [ ]:
names = await names_experiment()

## Exercises

1. Explain in your own words how the three transports differ in where the server runs and in who shares its data.

2. Measure how long the stdio transport takes to connect if the server imports nothing heavy. Where do the four seconds come from?

3. Repeat the test of the two receptionists with stdio, but making both clients use the same server process. Can it be done with the SDK's `Client`?

4. Add a third receptionist. Does anything change with HTTP? And with stdio?

5. Stop the HTTP server in the middle of the test and launch it again. What happens to the appointments already booked? What would be needed so that they weren't lost?

6. With the host without namespaces, print the list of tools the model receives. How many `search` tools are there? Which server does each call go to?

7. Change the host so that, when connecting, it rejects a tool with a name already in use, with a warning. How many tasks does the agent get right?

8. Try another form of namespace: leave the name as it is and add the server at the beginning of the description. Is it enough for the model to tell the two tools apart?

9. Connect chapter 4's agent to the clinic server over HTTP, instead of in memory, and run the chapter 1 tasks that its tools cover. Do the results change?

10. **Project.** Write a one-page recommendation for the clinic: which transport to use for each server (the schedule, the supplier's pharmacy and billing) and why, with the chapter's figures.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [MCP transports](https://modelcontextprotocol.io/specification) in the specification and [Python SDK](https://github.com/modelcontextprotocol/python-sdk). Northside Veterinary Clinic and its data are fictional. Original code and experiments.